In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. 数据加载与清洗
def load_data():
    try:
        # 加载标签数据
        tags = pd.read_csv('tags.csv')
        # 加载电影链接数据（用于后续获取电影名称）
        links = pd.read_csv('links.csv')
        # 数据清洗：去除重复的用户-电影标签记录
        tags = tags.drop_duplicates(subset=['userId', 'movieId'])

        # 尝试加载 movies.csv（用于显示电影名称）
        try:
            movies = pd.read_csv('movies.csv')
            merged = pd.merge(tags, movies[['movieId', 'title']], on='movieId', how='left')
        except FileNotFoundError:
            print("警告：未找到 movies.csv 文件，将无法显示电影名称")
            merged = tags.copy()

        return merged
    except FileNotFoundError:
        print("错误：找不到数据文件，请检查 links.csv、tags.csv 是否存在")
        return None

# 2. 创建用户-电影矩阵（标签计数作为评分）
def create_matrix(data):
    if data is None:
        return None, None

    # 创建用户-电影矩阵
    matrix = data.pivot_table(index='userId', columns='movieId', values='tag', aggfunc='count').fillna(0)
    # 获取电影名称映射
    movie_titles = data[['movieId', 'title']].drop_duplicates().set_index('movieId')

    return matrix, movie_titles

# 3. 计算用户相似度
def calculate_similarity(matrix):
    if matrix is None:
        return None
    return cosine_similarity(matrix)

# 4. 查找相似用户
def find_similar_users(user_id, matrix, similarity_matrix, k=5):
    if similarity_matrix is None:
        return []

    try:
        user_index = matrix.index.get_loc(user_id)  # 获取用户在矩阵中的索引
        sim_scores = similarity_matrix[user_index]
        similar_indices = sim_scores.argsort()[-k - 1:-1][::-1]
        similar_users = [matrix.index[idx] for idx in similar_indices if idx != user_index]
        return similar_users
    except KeyError:
        print(f"错误：用户 ID {user_id} 不存在于数据中")
        return []

# 5. 电影推荐
def recommend_movies(target_user, matrix, movie_titles, similarity_matrix, init_k=5, n=10):
    if matrix is None or movie_titles is None or similarity_matrix is None:
        return pd.DataFrame()

    if target_user not in matrix.index:
        print(f"错误：用户 ID {target_user} 不存在于数据中")
        return pd.DataFrame()

    user_movies = matrix.columns[matrix.loc[target_user] > 0]
    k = init_k
    while k >= 1:
        similar_users = find_similar_users(target_user, matrix, similarity_matrix, k)
        valid_similar_users = [user for user in similar_users if user in matrix.index]
        if valid_similar_users:
            neighbor_ratings = matrix.loc[valid_similar_users]
            recommendations = pd.DataFrame(neighbor_ratings.mean(axis=0), columns=['predicted_rating'])
            recommendations = recommendations[~recommendations.index.isin(user_movies)]
            recommendations = recommendations.sort_values('predicted_rating', ascending=False).head(n)
            recommendations = recommendations.join(movie_titles, on='movieId', how='left')
            return recommendations[['movieId', 'title', 'predicted_rating']]
        k -= 1
    print("无法找到任何相似用户")
    return pd.DataFrame()

# 主程序
if __name__ == "__main__":
    data = load_data()
    if data is None:
        exit()

    valid_user_ids = data['userId'].unique()
    print(f"有效用户 ID 范围：{min(valid_user_ids)} - {max(valid_user_ids)}")

    user_item_matrix, movie_titles = create_matrix(data)
    if user_item_matrix is None:
        exit()

    user_similarity = calculate_similarity(user_item_matrix)
    if user_similarity is None:
        exit()

    while True:
        try:
            target_user = int(input("请输入要推荐的用户 ID: "))
            if target_user in valid_user_ids:
                break
            else:
                print(f"错误：用户 ID {target_user} 不存在，请重新输入（有效范围：{min(valid_user_ids)} - {max(valid_user_ids)}）")
        except ValueError:
            print("输入无效，请输入整数")

    recommendations = recommend_movies(target_user, user_item_matrix, movie_titles, user_similarity)

    if not recommendations.empty:
        print(f"\n为用户 {target_user} 推荐的电影：")
        print(recommendations[['movieId', 'title', 'predicted_rating']].to_string(index=False))
    else:
        print("无法生成推荐")

有效用户 ID 范围：2 - 610


请输入要推荐的用户 ID:  2


KeyError: "['movieId'] not in index"

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. 读取数据并预处理
ratings = pd.read_csv('ratings.csv')
tags = pd.read_csv('tags.csv')

# 数据清洗：删除重复评分（保留同一用户对同一电影的最新评分）
ratings = ratings.sort_values('timestamp').drop_duplicates(subset=['userId', 'movieId'], keep='last')

# 2. 构建用户 - 电影评分矩阵（基于评分）
user_movie_matrix = ratings.pivot_table(
    index='userId', columns='movieId', values='rating', fill_value=0
)

# 3. 计算物品（电影）之间的余弦相似性
item_similarity = cosine_similarity(user_movie_matrix.T)

# 4. 构建电影标签特征矩阵（基于标签）
tag_matrix = tags.groupby(['movieId', 'tag']).size().unstack(fill_value=0)
tag_similarity = cosine_similarity(tag_matrix)

# 5. 综合评分和标签的相似性（加权融合）
combined_similarity = 0.7 * item_similarity + 0.3 * tag_similarity


def recommend_movies(target_user_id, num_recommendations=5):
    # 获取目标用户的评分记录
    user_ratings = user_movie_matrix.loc[target_user_id]

    # 找到用户未评分的电影
    unrated_movies = user_ratings[user_ratings == 0].index

    if not unrated_movies.empty:
        # 计算未评分电影与已评分电影的综合相似性
        movie_scores = {}
        for movie in unrated_movies:
            # 获取该电影与所有其他电影的相似性
            sim_scores = combined_similarity[movie]

            # 计算加权评分（相似性 * 评分）
            weighted_scores = sim_scores * user_ratings

            # 综合评分
            total_score = np.sum(weighted_scores) / np.sum(np.abs(sim_scores))
            movie_scores[movie] = total_score

        # 按评分排序并返回推荐
        sorted_movies = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_movies[:num_recommendations]
    else:
        return "用户已评分所有电影，无法推荐"


# 示例：为用户ID = 1推荐电影
recommendations = recommend_movies(target_user_id=1)
print("推荐电影：")
for movie_id, score in recommendations:
    print(f"电影ID: {movie_id}, 预测评分: {score:.2f}")

ValueError: operands could not be broadcast together with shapes (9724,9724) (1572,1572) 

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 读取数据
ratings = pd.read_csv('ratings.csv')
tags = pd.read_csv('tags.csv')
movies = pd.read_csv('movies.csv')

# 数据清洗：删除重复评分（保留同一用户对同一电影的最新评分）
ratings = ratings.sort_values('timestamp').drop_duplicates(subset=['userId', 'movieId'], keep='last')

# 构建用户-电影评分矩阵（基于评分）
user_movie_matrix = ratings.pivot_table(
    index='userId', columns='movieId', values='rating', fill_value=0
)

# 计算物品（电影）之间的余弦相似性
item_similarity = cosine_similarity(user_movie_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)

# 构建电影标签特征矩阵（基于标签）
tag_matrix = tags.groupby(['movieId', 'tag']).size().unstack(fill_value=0)
tag_similarity = cosine_similarity(tag_matrix)
tag_similarity_df = pd.DataFrame(tag_similarity, index=tag_matrix.index, columns=tag_matrix.index)

# 统一索引
common_movies = list(set(item_similarity_df.index).intersection(set(tag_similarity_df.index)))
item_similarity_common = item_similarity_df.loc[common_movies, common_movies]
tag_similarity_common = tag_similarity_df.loc[common_movies, common_movies]

# 综合评分和标签的相似性（加权融合）
combined_similarity = 0.7 * item_similarity_common + 0.3 * tag_similarity_common


def recommend_movies(target_user_id, num_recommendations=5):
    # 获取目标用户的评分记录
    user_ratings = user_movie_matrix.loc[target_user_id]

    # 找到用户未评分的电影
    unrated_movies = user_ratings[user_ratings == 0].index
    unrated_movies_common = [movie for movie in unrated_movies if movie in common_movies]

    if not unrated_movies_common:
        return "用户已评分所有电影，无法推荐"

    # 计算未评分电影与已评分电影的综合相似性
    movie_scores = {}
    for movie in unrated_movies_common:
        # 获取该电影与所有其他电影的相似性
        sim_scores = combined_similarity[movie]

        # 计算加权评分（相似性 * 评分）
        weighted_scores = sim_scores * user_ratings[common_movies]

        # 综合评分
        total_score = np.sum(weighted_scores) / np.sum(np.abs(sim_scores))
        movie_scores[movie] = total_score

    # 按评分排序并返回推荐
    sorted_movies = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)
    top_movies = sorted_movies[:num_recommendations]

    # 获取电影名称
    movie_names = []
    for movie_id, score in top_movies:
        movie_name = movies[movies['movieId'] == movie_id]['title'].values[0]
        movie_names.append(f"{movie_name} (预测评分: {score:.2f})")

    return movie_names


# 示例：为用户ID=1推荐电影
recommendations = recommend_movies(target_user_id=1)
print("推荐电影：")
for movie in recommendations:
    print(movie)
    

推荐电影：
Billabong Odyssey (2003) (预测评分: 0.90)
Reefer Madness: The Movie Musical (2005) (预测评分: 0.69)
Terminator 2: Judgment Day (1991) (预测评分: 0.69)
Arachnophobia (1990) (预测评分: 0.69)
War of the Worlds, The (1953) (预测评分: 0.68)


In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 读取数据
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

# 数据清洗：删除重复评分（保留同一用户对同一电影的最新评分）
ratings = ratings.sort_values('timestamp').drop_duplicates(subset=['userId', 'movieId'], keep='last')

# 构建用户 - 电影评分矩阵
user_movie_matrix = ratings.pivot_table(
    index='userId', columns='movieId', values='rating', fill_value=0
)

# 计算用户之间的余弦相似性
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)


def find_top_k_similar_users(user_id, k):
    """
    找到与指定用户最相似的 k 个用户
    """
    similar_users = user_similarity_df.loc[user_id].sort_values(ascending=False)
    top_k_users = similar_users.drop(user_id).head(k).index
    return top_k_users


def recommend_movies_for_user(user_id, k=5, num_recommendations=5):
    """
    为指定用户推荐电影
    """
    top_k_users = find_top_k_similar_users(user_id, k)
    target_user_ratings = user_movie_matrix.loc[user_id]
    unrated_movies = target_user_ratings[target_user_ratings == 0].index

    movie_scores = {}
    for movie in unrated_movies:
        total_score = 0
        similarity_sum = 0
        for similar_user in top_k_users:
            similar_user_rating = user_movie_matrix.loc[similar_user, movie]
            if similar_user_rating > 0:
                similarity = user_similarity_df.loc[user_id, similar_user]
                total_score += similarity * similar_user_rating
                similarity_sum += similarity
        if similarity_sum > 0:
            movie_scores[movie] = total_score / similarity_sum

    sorted_movies = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)
    top_movies = sorted_movies[:num_recommendations]

    # 获取电影名称
    movie_names = []
    for movie_id, score in top_movies:
        movie_name = movies[movies['movieId'] == movie_id]['title'].values[0]
        movie_names.append(f"{movie_name} (预测评分: {score:.2f})")

    return movie_names


# 为每个用户推荐电影
all_users = user_movie_matrix.index
for user in all_users:
    recommendations = recommend_movies_for_user(user)
    print(f"为用户 {user} 推荐的电影：")
    for movie in recommendations:
        print(movie)
    print()
    

为用户 1 推荐的电影：
Ref, The (1994) (预测评分: 5.00)
Blade Runner (1982) (预测评分: 5.00)
Wallace & Gromit: The Best of Aardman Animation (1996) (预测评分: 5.00)
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964) (预测评分: 5.00)
Godfather, The (1972) (预测评分: 5.00)

为用户 2 推荐的电影：
Seven (a.k.a. Se7en) (1995) (预测评分: 5.00)
Pulp Fiction (1994) (预测评分: 5.00)
Trainspotting (1996) (预测评分: 5.00)
Clockwork Orange, A (1971) (预测评分: 5.00)
Godfather: Part II, The (1974) (预测评分: 5.00)

为用户 3 推荐的电影：
Seven (a.k.a. Se7en) (1995) (预测评分: 5.00)
Usual Suspects, The (1995) (预测评分: 5.00)
Bottle Rocket (1996) (预测评分: 5.00)
Apollo 13 (1995) (预测评分: 5.00)
Clerks (1994) (预测评分: 5.00)

为用户 4 推荐的电影：
My Man Godfrey (1936) (预测评分: 5.00)
Ice Storm, The (1997) (预测评分: 5.00)
Boogie Nights (1997) (预测评分: 5.00)
Deconstructing Harry (1997) (预测评分: 5.00)
Player, The (1992) (预测评分: 5.00)

为用户 5 推荐的电影：
Leaving Las Vegas (1995) (预测评分: 5.00)
French Kiss (1995) (预测评分: 5.00)
Independence Day (a.k.a. ID4) (1996) (预测评分: 5.00)
Silence of the L

KeyboardInterrupt: 

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 读取数据
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

# 数据清洗：删除重复评分（保留同一用户对同一电影的最新评分）
ratings = ratings.sort_values('timestamp').drop_duplicates(subset=['userId', 'movieId'], keep='last')

# 构建用户 - 电影评分矩阵
user_movie_matrix = ratings.pivot_table(
    index='userId', columns='movieId', values='rating', fill_value=0
)

# 计算用户之间的余弦相似性
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)


def find_top_k_similar_users(user_id, k):
    """
    找到与指定用户最相似的 k 个用户
    """
    similar_users = user_similarity_df.loc[user_id].sort_values(ascending=False)
    top_k_users = similar_users.drop(user_id).head(k).index
    return top_k_users


def recommend_movies_for_user(user_id, k=5, num_recommendations=5):
    """
    为指定用户推荐电影
    """
    top_k_users = find_top_k_similar_users(user_id, k)
    target_user_ratings = user_movie_matrix.loc[user_id]
    unrated_movies = target_user_ratings[target_user_ratings == 0].index

    movie_scores = {}
    for movie in unrated_movies:
        total_score = 0
        similarity_sum = 0
        for similar_user in top_k_users:
            similar_user_rating = user_movie_matrix.loc[similar_user, movie]
            if similar_user_rating > 0:
                similarity = user_similarity_df.loc[user_id, similar_user]
                total_score += similarity * similar_user_rating
                similarity_sum += similarity
        if similarity_sum > 0:
            movie_scores[movie] = total_score / similarity_sum

    sorted_movies = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)
    top_movies = sorted_movies[:num_recommendations]

    # 获取电影名称
    movie_names = []
    for movie_id, _ in top_movies:
        movie_name = movies[movies['movieId'] == movie_id]['title'].values[0]
        movie_names.append(f"电影编号: {movie_id}, 电影名称: {movie_name}")

    return movie_names


# 让用户输入要查询的用户 ID
while True:
    try:
        user_id = int(input("请输入要查询的用户 ID（输入 -1 退出）："))
        if user_id == -1:
            break
        if user_id not in user_movie_matrix.index:
            print("该用户 ID 不存在，请重新输入。")
            continue
        recommendations = recommend_movies_for_user(user_id)
        print(f"为用户 {user_id} 推荐的电影：")
        for movie in recommendations:
            print(movie)
        print()
    except ValueError:
        print("输入无效，请输入有效的整数用户 ID。")
    

请输入要查询的用户 ID（输入 -1 退出）： 1


为用户 1 推荐的电影：
电影编号: 514, 电影名称: Ref, The (1994)
电影编号: 541, 电影名称: Blade Runner (1982)
电影编号: 720, 电影名称: Wallace & Gromit: The Best of Aardman Animation (1996)
电影编号: 750, 电影名称: Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)
电影编号: 858, 电影名称: Godfather, The (1972)



请输入要查询的用户 ID（输入 -1 退出）： 2


为用户 2 推荐的电影：
电影编号: 47, 电影名称: Seven (a.k.a. Se7en) (1995)
电影编号: 296, 电影名称: Pulp Fiction (1994)
电影编号: 778, 电影名称: Trainspotting (1996)
电影编号: 1206, 电影名称: Clockwork Orange, A (1971)
电影编号: 1221, 电影名称: Godfather: Part II, The (1974)



请输入要查询的用户 ID（输入 -1 退出）： 12


为用户 12 推荐的电影：
电影编号: 265, 电影名称: Like Water for Chocolate (Como agua para chocolate) (1992)
电影编号: 293, 电影名称: Léon: The Professional (a.k.a. The Professional) (Léon) (1994)
电影编号: 509, 电影名称: Piano, The (1993)
电影编号: 904, 电影名称: Rear Window (1954)
电影编号: 912, 电影名称: Casablanca (1942)



请输入要查询的用户 ID（输入 -1 退出）： -1


In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

#读取数据
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

#读取数据后的前 5 行
print("ratings 数据前 5 行：")
print(ratings.head().to_csv(sep='\t', na_rep='nan'))
print("movies 数据前 5 行：")
print(movies.head().to_csv(sep='\t', na_rep='nan'))

ratings 数据前 5 行：
	userId	movieId	rating	timestamp
0	1	1	4.0	964982703
1	1	3	4.0	964981247
2	1	6	4.0	964982224
3	1	47	5.0	964983815
4	1	50	5.0	964982931

movies 数据前 5 行：
	movieId	title	genres
0	1	Toy Story (1995)	Adventure|Animation|Children|Comedy|Fantasy
1	2	Jumanji (1995)	Adventure|Children|Fantasy
2	3	Grumpier Old Men (1995)	Comedy|Romance
3	4	Waiting to Exhale (1995)	Comedy|Drama|Romance
4	5	Father of the Bride Part II (1995)	Comedy



In [7]:
# 数据清洗：删除重复评分（保留同一用户对同一电影的最新评分）
ratings = ratings.sort_values('timestamp').drop_duplicates(subset=['userId', 'movieId'], keep='last')
ratings

,userId,movieId,rating,timestamp
66719,429,595,5.0,828124615
66716,429,588,5.0,828124615
66717,429,590,5.0,828124615
66718,429,592,5.0,828124615
66712,429,432,3.0,828124615
...,...,...,...,...
81475,514,187031,2.5,1537674927
81477,514,187595,3.0,1537674946
81336,514,5247,2.5,1537757040
81335,514,5246,1.5,1537757059


In [11]:
#构建用户 - 电影评分矩阵
user_movie_matrix = ratings.pivot_table(
    index='userId', columns='movieId', values='rating', fill_value=0
)
#展示用户-电影评分矩阵前5行、前5列
print("用户 - 电影评分矩阵前5行，前5列：")
print(user_movie_matrix.head(5).iloc[:, :5].to_csv(sep='\t', na_rep='nan'))

用户 - 电影评分矩阵前5行，前5列：
userId	1	2	3	4	5
1	4.0	0.0	4.0	0.0	0.0
2	0.0	0.0	0.0	0.0	0.0
3	0.0	0.0	0.0	0.0	0.0
4	0.0	0.0	0.0	0.0	0.0
5	4.0	0.0	0.0	0.0	0.0



In [12]:
#计算用户之间的余弦相似性
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)

#展示前5个用户间相似性
print("前5个用户间相似性：")
print(user_similarity_df.head().iloc[:, :5].to_csv(sep='\t', na_rep='nan'))

前5个用户间相似性：
userId	1	2	3	4	5
1	1.0	0.027282865283323236	0.05972026184426368	0.1943947687436415	0.12907989044170284
2	0.027282865283323236	0.9999999999999999	0.0	0.0037258658575476124	0.016614444008872527
3	0.05972026184426368	0.0	1.0	0.002251389028959687	0.005019716006155572
4	0.1943947687436415	0.0037258658575476124	0.002251389028959687	1.0	0.12865890160687454
5	0.12907989044170284	0.016614444008872527	0.005019716006155572	0.12865890160687454	1.0



In [5]:
#找到与指定用户最相似的 k 个用户
def find_top_k_similar_users(user_id, k):
    similar_users = user_similarity_df.loc[user_id].sort_values(ascending=False)
    top_k_users = similar_users.drop(user_id).head(k).index
    return top_k_users

#建立推荐电影模型
def recommend_movies_for_user(user_id, k=5, num_recommendations=5):
    top_k_users = find_top_k_similar_users(user_id, k)
    target_user_ratings = user_movie_matrix.loc[user_id]
    unrated_movies = target_user_ratings[target_user_ratings == 0].index

    movie_scores = {}
    for movie in unrated_movies:
        total_score = 0
        similarity_sum = 0
        for similar_user in top_k_users:
            similar_user_rating = user_movie_matrix.loc[similar_user, movie]
            if similar_user_rating > 0:
                similarity = user_similarity_df.loc[user_id, similar_user]
                total_score += similarity * similar_user_rating
                similarity_sum += similarity
        if similarity_sum > 0:
            movie_scores[movie] = total_score / similarity_sum

    sorted_movies = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)
    top_movies = sorted_movies[:num_recommendations]

    #获取电影名称
    movie_names = []
    for movie_id, _ in top_movies:
        movie_name = movies[movies['movieId'] == movie_id]['title'].values[0]
        movie_names.append(f"推荐电影：{movie_id} {movie_name}")
    return movie_names

In [6]:
#输入要查询的用户 ID，输出推荐电影
while True:
    try:
        user_id = int(input("请输入要查询的用户 ID（输入 -1 退出）："))
        if user_id == -1:
            break
        if user_id not in user_movie_matrix.index:
            print("该用户 ID 不存在，请重新输入。")
            continue
        recommendations = recommend_movies_for_user(user_id)
        print(f"为用户 {user_id} 推荐的电影：")
        for movie in recommendations:
            print(movie)
        print()
    except ValueError:
        print("输入无效，请输入有效的整数用户 ID。")

请输入要查询的用户 ID（输入 -1 退出）： 1


NameError: name 'user_movie_matrix' is not defined

In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 读取数据
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

# 展示读取数据后的前 5 行
print("ratings 数据前 5 行：")
print(ratings.head().to_csv(sep='\t', na_rep='nan'))
print("movies 数据前 5 行：")
print(movies.head().to_csv(sep='\t', na_rep='nan'))

# 数据清洗：删除重复评分（保留同一用户对同一电影的最新评分）
ratings = ratings.sort_values('timestamp').drop_duplicates(subset=['userId', 'movieId'], keep='last')

# 构建用户 - 电影评分矩阵
user_movie_matrix = ratings.pivot_table(
    index='userId', columns='movieId', values='rating', fill_value=0
)

# 展示用户 - 电影评分矩阵前 5 行，前 20 列
print("用户 - 电影评分矩阵前 5 行，前 20 列：")
print(user_movie_matrix.head(5).iloc[:, :20].to_csv(sep='\t', na_rep='nan'))

# 计算用户之间的余弦相似性
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)

# 展示部分用户间相似性（前 5 个用户之间）
print("部分用户间相似性（前 5 个用户之间）：")
print(user_similarity_df.head().iloc[:, :5].to_csv(sep='\t', na_rep='nan'))


def find_top_k_similar_users(user_id, k):
    """
    找到与指定用户最相似的 k 个用户
    """
    similar_users = user_similarity_df.loc[user_id].sort_values(ascending=False)
    top_k_users = similar_users.drop(user_id).head(k).index
    return top_k_users



ratings 数据前 5 行：
	userId	movieId	rating	timestamp
0	1	1	4.0	964982703
1	1	3	4.0	964981247
2	1	6	4.0	964982224
3	1	47	5.0	964983815
4	1	50	5.0	964982931

movies 数据前 5 行：
	movieId	title	genres
0	1	Toy Story (1995)	Adventure|Animation|Children|Comedy|Fantasy
1	2	Jumanji (1995)	Adventure|Children|Fantasy
2	3	Grumpier Old Men (1995)	Comedy|Romance
3	4	Waiting to Exhale (1995)	Comedy|Drama|Romance
4	5	Father of the Bride Part II (1995)	Comedy

用户 - 电影评分矩阵前 5 行，前 20 列：
userId	1	2	3	4	5	6	7	8	9	10	11	12	13	14	15	16	17	18	19	20
1	4.0	0.0	4.0	0.0	0.0	4.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
2	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
3	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
4	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
5	4.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0

部分用户间相似性（前 5 个用户之间）：
userId	1	2	3	4	5
1	1.0	0.027282865283323236

In [8]:
#找到与指定用户最相似的 k 个用户
def find_top_k_similar_users(user_id, k):
    similar_users = user_similarity_df.loc[user_id].sort_values(ascending=False)
    top_k_users = similar_users.drop(user_id).head(k).index
    return top_k_users

#建立推荐电影模型
def recommend_movies_for_user(user_id, k=5, num_recommendations=5):
    top_k_users = find_top_k_similar_users(user_id, k)
    target_user_ratings = user_movie_matrix.loc[user_id]
    unrated_movies = target_user_ratings[target_user_ratings == 0].index

    movie_scores = {}
    for movie in unrated_movies:
        total_score = 0
        similarity_sum = 0
        for similar_user in top_k_users:
            similar_user_rating = user_movie_matrix.loc[similar_user, movie]
            if similar_user_rating > 0:
                similarity = user_similarity_df.loc[user_id, similar_user]
                total_score += similarity * similar_user_rating
                similarity_sum += similarity
        if similarity_sum > 0:
            movie_scores[movie] = total_score / similarity_sum

    sorted_movies = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)
    top_movies = sorted_movies[:num_recommendations]

    #获取电影名称
    movie_names = []
    for movie_id, _ in top_movies:
        movie_name = movies[movies['movieId'] == movie_id]['title'].values[0]
        movie_names.append(f"推荐电影：{movie_id} {movie_name}")

    return movie_names

In [9]:
#让用户输入要查询的用户 ID
while True:
    try:
        user_id = int(input("请输入要查询的用户 ID（输入 -1 退出）："))
        if user_id == -1:
            break
        if user_id not in user_movie_matrix.index:
            print("该用户 ID 不存在，请重新输入。")
            continue
        recommendations = recommend_movies_for_user(user_id)
        print(f"为用户 {user_id} 推荐的电影：")
        for movie in recommendations:
            print(movie)
        print()
    except ValueError:
        print("输入无效，请输入有效的整数用户 ID。")

请输入要查询的用户 ID（输入 -1 退出）： 1


为用户 1 推荐的电影：
推荐电影：514 Ref, The (1994)
推荐电影：541 Blade Runner (1982)
推荐电影：720 Wallace & Gromit: The Best of Aardman Animation (1996)
推荐电影：750 Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)
推荐电影：858 Godfather, The (1972)



请输入要查询的用户 ID（输入 -1 退出）： 12


为用户 12 推荐的电影：
推荐电影：265 Like Water for Chocolate (Como agua para chocolate) (1992)
推荐电影：293 Léon: The Professional (a.k.a. The Professional) (Léon) (1994)
推荐电影：509 Piano, The (1993)
推荐电影：904 Rear Window (1954)
推荐电影：912 Casablanca (1942)



请输入要查询的用户 ID（输入 -1 退出）： -1


In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 读取数据
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

# 展示读取数据后的前 5 行
print("ratings 数据前 5 行：")
print(ratings.head().to_csv(sep='\t', na_rep='nan'))
print("movies 数据前 5 行：")
print(movies.head().to_csv(sep='\t', na_rep='nan'))

ratings 数据前 5 行：
	userId	movieId	rating	timestamp
0	1	1	4.0	964982703
1	1	3	4.0	964981247
2	1	6	4.0	964982224
3	1	47	5.0	964983815
4	1	50	5.0	964982931

movies 数据前 5 行：
	movieId	title	genres
0	1	Toy Story (1995)	Adventure|Animation|Children|Comedy|Fantasy
1	2	Jumanji (1995)	Adventure|Children|Fantasy
2	3	Grumpier Old Men (1995)	Comedy|Romance
3	4	Waiting to Exhale (1995)	Comedy|Drama|Romance
4	5	Father of the Bride Part II (1995)	Comedy



In [2]:
# 数据清洗：删除重复评分（保留同一用户对同一电影的最新评分）
ratings = ratings.sort_values('timestamp').drop_duplicates(subset=['userId', 'movieId'], keep='last')

# 构建用户 - 电影评分矩阵
user_movie_matrix = ratings.pivot_table(
    index='userId', columns='movieId', values='rating', fill_value=0
)

# 展示用户 - 电影评分矩阵前 5 行，前 20 列
print("用户 - 电影评分矩阵前 5 行，前 20 列：")
print(user_movie_matrix.head(5).iloc[:, :20].to_csv(sep='\t', na_rep='nan'))


用户 - 电影评分矩阵前 5 行，前 20 列：
userId	1	2	3	4	5	6	7	8	9	10	11	12	13	14	15	16	17	18	19	20
1	4.0	0.0	4.0	0.0	0.0	4.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
2	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
3	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
4	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
5	4.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0



In [3]:
# 计算用户之间的余弦相似性
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)

# 展示部分用户间相似性（前 5 个用户之间）
print("部分用户间相似性（前 5 个用户之间）：")
print(user_similarity_df.head().iloc[:, :5].to_csv(sep='\t', na_rep='nan'))


部分用户间相似性（前 5 个用户之间）：
userId	1	2	3	4	5
1	1.0	0.027282865283323236	0.05972026184426368	0.1943947687436415	0.12907989044170284
2	0.027282865283323236	0.9999999999999999	0.0	0.0037258658575476124	0.016614444008872527
3	0.05972026184426368	0.0	1.0	0.002251389028959687	0.005019716006155572
4	0.1943947687436415	0.0037258658575476124	0.002251389028959687	1.0	0.12865890160687454
5	0.12907989044170284	0.016614444008872527	0.005019716006155572	0.12865890160687454	1.0



In [4]:
def find_top_k_similar_users(user_id, k):
    """
    找到与指定用户最相似的 k 个用户
    """
    similar_users = user_similarity_df.loc[user_id].sort_values(ascending=False)
    top_k_users = similar_users.drop(user_id).head(k).index
    return top_k_users

In [5]:
#建立推荐电影模型
def recommend_movies_for_user(user_id, k=5, num_recommendations=5):
    top_k_users = find_top_k_similar_users(user_id, k)
    target_user_ratings = user_movie_matrix.loc[user_id]
    unrated_movies = target_user_ratings[target_user_ratings == 0].index

    movie_scores = {}
    for movie in unrated_movies:
        total_score = 0
        similarity_sum = 0
        for similar_user in top_k_users:
            similar_user_rating = user_movie_matrix.loc[similar_user, movie]
            if similar_user_rating > 0:
                similarity = user_similarity_df.loc[user_id, similar_user]
                total_score += similarity * similar_user_rating
                similarity_sum += similarity
        if similarity_sum > 0:
            movie_scores[movie] = total_score / similarity_sum

    sorted_movies = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)
    top_movies = sorted_movies[:num_recommendations]

    #获取电影名称
    movie_names = []
    for movie_id, _ in top_movies:
        movie_name = movies[movies['movieId'] == movie_id]['title'].values[0]
        movie_names.append(f"推荐电影：{movie_id} {movie_name}")

    return movie_names

In [6]:
#让用户输入要查询的用户 ID
while True:
    try:
        user_id = int(input("请输入要查询的用户 ID（输入 -1 退出）："))
        if user_id == -1:
            break
        if user_id not in user_movie_matrix.index:
            print("该用户 ID 不存在，请重新输入。")
            continue
        recommendations = recommend_movies_for_user(user_id)
        print(f"为用户 {user_id} 推荐的电影：")
        for movie in recommendations:
            print(movie)
        print()
    except ValueError:
        print("输入无效，请输入有效的整数用户 ID。")

请输入要查询的用户 ID（输入 -1 退出）： 1


为用户 1 推荐的电影：
推荐电影：514 Ref, The (1994)
推荐电影：541 Blade Runner (1982)
推荐电影：720 Wallace & Gromit: The Best of Aardman Animation (1996)
推荐电影：750 Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)
推荐电影：858 Godfather, The (1972)



请输入要查询的用户 ID（输入 -1 退出）： 12


为用户 12 推荐的电影：
推荐电影：265 Like Water for Chocolate (Como agua para chocolate) (1992)
推荐电影：293 Léon: The Professional (a.k.a. The Professional) (Léon) (1994)
推荐电影：509 Piano, The (1993)
推荐电影：904 Rear Window (1954)
推荐电影：912 Casablanca (1942)



请输入要查询的用户 ID（输入 -1 退出）： -1
